# 商品客户购买频次分析

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import URL
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv(r"D:\my_project\ecommerce-business-analytics\.env")
engine = create_engine(URL.create(
    "mysql+pymysql",
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 3306)),
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    query={"charset": "utf8mb4"}
))

In [ ]:
# 1.准备数据
data = pd.read_sql('select * from v_sales_transactions where invoice_date >= "2010-11-01" and invoice_date < "2011-12-01" and customer_id is not null', engine)

In [ ]:
# 2.分别计算出每件商品的购买人数
purchase = data.groupby('stock_code')['customer_id'].nunique().reset_index(name = 'purchase_total_count')

In [ ]:
# 3.同一客户购买同一商品，出现在至少两张不同发票中，算作该商品的重复购买客户
repeat = data.groupby(['stock_code', 'customer_id'])['invoice'].nunique().reset_index(name = 'repeat_count')

repeat_purchase1 = repeat.query('repeat_count > 1')
repeat_purchase = repeat_purchase1.groupby('stock_code')['customer_id'].nunique().reset_index(name = 'customer_count')
repeat_purchase

In [ ]:
# 4.将两张表通过merge函数和为一张
finall_table = purchase.merge(repeat_purchase, how='left', on='stock_code')
# 5.填充customer_count列合并后的缺失值为0
finall_table['customer_count'] =finall_table['customer_count'].fillna(0)
# 6.计算复购率，使用百分比并且保留两位小数
finall_table['repeat_ratio'] = ((finall_table['customer_count'] / finall_table['purchase_total_count']) * 100).round(2)
# 7.筛选目标列
finall_table = finall_table[[
    'stock_code',
'purchase_total_count',
'customer_count',
'repeat_ratio'
]]
finall_table

### 哪些商品值得优先做复购推广？

优先挑选**购买客户多、重复购买客户也多，而且重复购买客户占比较高**的商品，做小规模复购推广。筛选时，先看观察期内有多少位可识别客户买过这件商品，再看其中有多少人至少在两张不同发票中买过它。在购买人数相近的商品中，优先测试重复购买客户占比较高的商品。只有一两位买家的商品，即使占比很高，也先留作观察。

对于入选商品，可以面向观察期内只买过该商品一次的客户，测试再次购买提醒。比较收到提醒和未收到提醒的两组客户在同一时间内再次购买该商品的比例，再决定是否扩大推广。

In [12]:
# 8.选出购买客户至少 100 人、其中重复购买客户至少 20 人的商品，再按重复购买客户占比从高到低取前 3 款
choose_table = finall_table.query('purchase_total_count >= 100 & customer_count >= 20').sort_values('repeat_ratio', ascending=False).head(3)
example = choose_table.merge(data, 'left', on='stock_code').drop_duplicates(subset='stock_code', keep='last')[['description',
               'stock_code',
               'purchase_total_count',
               'customer_count',
               'repeat_ratio'                                                                                           ]]
example

,description,stock_code,purchase_total_count,customer_count,repeat_ratio
1281,LUNCH BAG RED RETROSPOT,20725,524,272.0,51.91
2343,LUNCH BAG SUKI DESIGN,22383,433,222.0,51.27
3161,LUNCH BAG WOODLAND,20726,364,186.0,51.10
